# 01. Reconstruct a collision
**Lectures 1–4 · Draft teaching exercise; use instructor-approved samples.**

Use a MadGraph-only e+e− → μ+μ− sample at fixed beam energy. The stored pair should exhaust this two-body final state. Here +z must be the incoming electron direction; confirm beam ordering with the instructor. No shower, ISR, or invisible particles for the conservation test.

Save your own copy before editing. Run cells from the top. Empty sample selections deliberately do nothing; choose IDs from the available-samples table.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from findingz.catalog import load_catalog
from findingz.delphes import default_run_root
from findingz.hypotheses import build_sample_library

library = build_sample_library(load_catalog(), default_run_root())
display(pd.DataFrame([{"sample_id": k, "label": s.label, "cross_section_pb": s.cross_section_pb,
                      "events": s.generated_events, "config": s.config} for k, s in library.items()]))
def get_sample(sample_id):
    if sample_id not in library:
        raise ValueError(f"Sample {sample_id!r} is unavailable. Choose an ID from the table above.")
    return library[sample_id]
def weights(frame):
    return pd.to_numeric(frame.get("weight", pd.Series(1., index=frame.index)))


In [ ]:
def four_vector(frame, prefix):
    pt, eta, phi, mass = [frame[prefix + "_" + x].to_numpy() for x in ("pt", "eta", "phi", "mass")]
    px, py, pz = pt*np.cos(phi), pt*np.sin(phi), pt*np.sinh(eta)
    return np.column_stack([np.sqrt(px*px+py*py+pz*pz+mass*mass), px, py, pz])
def pair_observables(frame):
    a, b = four_vector(frame, "l1"), four_vector(frame, "l2")
    total = a+b
    mass = np.sqrt(np.maximum(total[:,0]**2 - (total[:,1:]**2).sum(axis=1), 0))
    negative = np.where((frame.l1_charge.to_numpy() < 0)[:,None], a, b)
    momentum = np.linalg.norm(negative[:,1:], axis=1)
    cosine = np.divide(negative[:,3], momentum, out=np.full_like(momentum,np.nan), where=momentum>0)
    return mass, np.hypot(total[:,1],total[:,2]), cosine, total


In [ ]:
sample_id = None  # choose a MadGraph-only ee_mumu sample
if sample_id:
    sample = get_sample(sample_id)
    frame = sample.load()
    mass, pair_pt, cosine, total = pair_observables(frame)
    energy = 2*float(sample.config["beam_energy_gev"])
    display(pd.DataFrame({"mll": mass, "pair_pt": pair_pt, "cos_theta_minus": cosine}).describe())
    print("Maximum |Epair - sqrt(s)| [GeV]:", np.max(abs(total[:,0]-energy)))
    print("Maximum pair three-momentum [GeV]:", np.max(np.linalg.norm(total[:,1:],axis=1)))
    fig, axes = plt.subplots(1,3,figsize=(12,3))
    for ax, value, label in zip(axes,[mass,pair_pt,cosine],["mll [GeV]","pair pT [GeV]","cos θ(mu−)"]):
        ax.hist(value,bins=30); ax.set_xlabel(label)
    plt.show()

## Questions and submission
1. Predict the mass and pair pT before plotting. Explain any numerical spread.
2. Why is a fixed-energy two-body mass histogram not an energy-scan line shape?
3. Which distribution contains dynamical information beyond energy–momentum conservation?
4. Extension: boost both four-vectors along z and check that invariant mass is unchanged.

Submit your edited notebook with figures, units, sample IDs, generation settings, and a short interpretation. Do not equate Monte Carlo event count with experimental luminosity.